In [ ]:
import os, time, copy, random, csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import torch.ao.quantization as tq   # quantization API

# reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Train device:", DEVICE)
print("Torch version:", torch.__version__)

# project folders
ROOT = "/content/edge_assignment"
Q1 = os.path.join(ROOT, "q1_quantization")
CKPT = os.path.join(Q1, "checkpoints")
RESULTS = os.path.join(Q1, "results")
for d in [ROOT, Q1, CKPT, RESULTS]:
    os.makedirs(d, exist_ok=True)


EPOCHS_FP32 = 15     # baseline training epochs (bump to 30+ for higher accuracy)
EPOCHS_QAT  = 5      # QAT fine-tuning epochs
BATCH = 128
CALIB_BATCHES = 20   #  training batches PTQ uses for calibration
QBACKEND = "fbgemm"  # x86 CPU backend (Colab is x86). ARM edge devices use "qnnpack"
torch.backends.quantized.engine = QBACKEND
print("Quantization backend:", QBACKEND)

Train device: cuda
Torch version: 2.14.1+cu130
Quantization backend: fbgemm


In [ ]:
# CIFAR-10 normalization constant
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

train_set = torchvision.datasets.CIFAR10("/content/data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10("/content/data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# calibration loader: a fixed subset of the TRAIN set, no augmentation
calib_set = torchvision.datasets.CIFAR10("/content/data", train=True, download=False, transform=test_tf)
calib_subset = Subset(calib_set, list(range(CALIB_BATCHES * BATCH)))
calib_loader = DataLoader(calib_subset, batch_size=BATCH, shuffle=False, num_workers=2)

CLASSES = train_set.classes
print("Train:", len(train_set), "Test:", len(test_set), "Calib imgs:", len(calib_subset))

Train: 50000 Test: 10000 Calib imgs: 2560


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.relu1 = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.skip_add = nn.quantized.FloatFunctional()  # quant-safe addition
        self.relu2 = nn.ReLU(inplace=False)

    def forward(self, x):
        identity = x
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = self.skip_add.add(out, identity)  # NOT out += identity
        out = self.relu2(out)
        return out

    def fuse_block(self, is_qat):
        fuse = tq.fuse_modules_qat if is_qat else tq.fuse_modules
        fuse(self, [["conv1", "bn1", "relu1"], ["conv2", "bn2"]], inplace=True)
        if self.downsample is not None:
            fuse(self.downsample, ["0", "1"], inplace=True)


class QuantResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.quant = tq.QuantStub()
        # CIFAR stem: 3x3 stride 1, no maxpool
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)
        self.relu  = nn.ReLU(inplace=False)
        self.layer1 = self._make_layer(64,  2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        self.dequant = tq.DeQuantStub()

    def _make_layer(self, planes, blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        layers = [BasicBlock(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.quant(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.dequant(x)
        return x

    def fuse_model(self, is_qat=False):
        fuse = tq.fuse_modules_qat if is_qat else tq.fuse_modules
        fuse(self, [["conv1", "bn1", "relu"]], inplace=True)
        for m in self.modules():
            if isinstance(m, BasicBlock):
                m.fuse_block(is_qat)

print(QuantResNet18())

QuantResNet18(
  (quant): QuantStub()
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu1): ReLU()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (skip_add): FloatFunctional(
        (activation_post_process): Identity()
      )
      (relu2): ReLU()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True

In [ ]:
def evaluate(model, loader, device):
    model.eval().to(device)
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

def model_size_mb(model, path):
    torch.save(model.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    return mb

@torch.no_grad()
def cpu_latency_ms(model, runs=100):
    # single-image latency on CPU, averaged
    model.eval().to("cpu")
    torch.set_num_threads(1)
    x = torch.randn(1, 3, 32, 32)
    for _ in range(10):      # warmup
        model(x)
    t0 = time.time()
    for _ in range(runs):
        model(x)
    return (time.time() - t0) / runs * 1000.0

@torch.no_grad()
def get_predictions(model, loader, device):
    # returns list of predicted class indices over the whole test set
    model.eval().to(device)
    preds = []
    for x, _ in loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
    return preds

In [ ]:
def train_model(model, loader, epochs, lr, device, tag="model"):
    model.to(device).train()
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        running = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            running += loss.item()
        sched.step()
        print(f"[{tag}] epoch {ep+1}/{epochs}  loss {running/len(loader):.3f}  lr {sched.get_last_lr()[0]:.4f}")
    return model

fp32 = QuantResNet18(num_classes=10)
fp32 = train_model(fp32, train_loader, EPOCHS_FP32, lr=0.1, device=DEVICE, tag="FP32")

fp32_path = os.path.join(CKPT, "fp32_baseline.pth")
torch.save(fp32.state_dict(), fp32_path)

acc = evaluate(fp32, test_loader, DEVICE)
print(f"\nFP32 test accuracy: {acc:.2f}%  | saved: {fp32_path}")

[FP32] epoch 1/15  loss 1.911  lr 0.0989
[FP32] epoch 2/15  loss 1.460  lr 0.0957
[FP32] epoch 3/15  loss 1.197  lr 0.0905
[FP32] epoch 4/15  loss 1.000  lr 0.0835
[FP32] epoch 5/15  loss 0.847  lr 0.0750
[FP32] epoch 6/15  loss 0.716  lr 0.0655
[FP32] epoch 7/15  loss 0.609  lr 0.0552
[FP32] epoch 8/15  loss 0.527  lr 0.0448
[FP32] epoch 9/15  loss 0.463  lr 0.0345
[FP32] epoch 10/15  loss 0.400  lr 0.0250
[FP32] epoch 11/15  loss 0.347  lr 0.0165
[FP32] epoch 12/15  loss 0.294  lr 0.0095
[FP32] epoch 13/15  loss 0.238  lr 0.0043
[FP32] epoch 14/15  loss 0.199  lr 0.0011
[FP32] epoch 15/15  loss 0.174  lr 0.0000

FP32 test accuracy: 90.32%  | saved: /content/edge_assignment/q1_quantization/checkpoints/fp32_baseline.pth


In [ ]:
# always start PTQ from a fresh copy of the trained FP32 weights, on CPU
ptq = QuantResNet18(num_classes=10)
ptq.load_state_dict(torch.load(fp32_path, map_location="cpu"))
ptq.to("cpu").eval()

# 1) fuse layers
ptq.fuse_model(is_qat=False)

# 2) set backend qconfig
ptq.qconfig = tq.get_default_qconfig(QBACKEND)

# 3) insert observers
tq.prepare(ptq, inplace=True)

# 4) calibrate: run calibration data so observers record activation ranges
with torch.no_grad():
    for x, _ in calib_loader:
        ptq(x)
print("Calibration done on", len(calib_subset), "images")

# 5) convert FP32 -> INT8
tq.convert(ptq, inplace=True)

# 6) save a deployable (self-contained) INT8 model via TorchScript
ptq_path = os.path.join(CKPT, "ptq_int8.pt")
torch.jit.save(torch.jit.script(ptq), ptq_path)

ptq_acc = evaluate(ptq, test_loader, "cpu")   # INT8 must run on CPU
print(f"PTQ-INT8 test accuracy: {ptq_acc:.2f}%  | saved: {ptq_path}")

/tmp/ipykernel_388/2274052180.py:13: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  tq.prepare(ptq, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__

Calibration done on 2560 images


/tmp/ipykernel_388/2274052180.py:22: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  tq.convert(ptq, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/modules/utils.py:78: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and t

PTQ-INT8 test accuracy: 90.27%  | saved: /content/edge_assignment/q1_quantization/checkpoints/ptq_int8.pt


In [ ]:
# start QAT from the pretrained FP32 weights (assignment requirement)
qat = QuantResNet18(num_classes=10)
qat.load_state_dict(torch.load(fp32_path, map_location="cpu"))
qat.to(DEVICE).train()

# fuse (QAT-mode fusion) + insert fake-quant modules
qat.fuse_model(is_qat=True)
qat.qconfig = tq.get_default_qat_qconfig(QBACKEND)
tq.prepare_qat(qat, inplace=True)

# fine-tune with a small learning rate
opt = optim.SGD(qat.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
crit = nn.CrossEntropyLoss()
for ep in range(EPOCHS_QAT):
    qat.train(); running = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = crit(qat(x), y)
        loss.backward()
        opt.step()
        running += loss.item()
    # late in training, freeze observers so stats stop moving (improves INT8 accuracy)
    if ep >= EPOCHS_QAT - 2:
        qat.apply(tq.disable_observer)
    print(f"[QAT] epoch {ep+1}/{EPOCHS_QAT}  loss {running/len(train_loader):.3f}")

# convert to INT8 (must happen on CPU)
qat.to("cpu").eval()
tq.convert(qat, inplace=True)

qat_path = os.path.join(CKPT, "qat_int8.pt")
torch.jit.save(torch.jit.script(qat), qat_path)

qat_acc = evaluate(qat, test_loader, "cpu")
print(f"QAT-INT8 test accuracy: {qat_acc:.2f}%  | saved: {qat_path}")

/tmp/ipykernel_388/273962922.py:9: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  tq.prepare_qat(qat, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super()._

[QAT] epoch 1/5  loss 0.170
[QAT] epoch 2/5  loss 0.162
[QAT] epoch 3/5  loss 0.159
[QAT] epoch 4/5  loss 0.155
[QAT] epoch 5/5  loss 0.154


/tmp/ipykernel_388/273962922.py:30: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  tq.convert(qat, inplace=True)


QAT-INT8 test accuracy: 90.69%  | saved: /content/edge_assignment/q1_quantization/checkpoints/qat_int8.pt


In [ ]:
def load_model(kind):
    """kind in {'fp32','ptq','qat'} -> returns a ready-to-infer model on CPU."""
    if kind == "fp32":
        m = QuantResNet18(num_classes=10)
        m.load_state_dict(torch.load(fp32_path, map_location="cpu"))
        return m.eval()
    elif kind == "ptq":
        return torch.jit.load(ptq_path).eval()
    elif kind == "qat":
        return torch.jit.load(qat_path).eval()
    raise ValueError(kind)

# quick check: predict one test image with each model
sample_x, sample_y = test_set[0]
sample_x = sample_x.unsqueeze(0)
for k in ["fp32", "ptq", "qat"]:
    m = load_model(k)
    p = m(sample_x).argmax(1).item()
    print(f"{k:4s} -> predicted {CLASSES[p]:10s} (true: {CLASSES[sample_y]})")

fp32 -> predicted cat        (true: cat)
ptq  -> predicted cat        (true: cat)


/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


qat  -> predicted cat        (true: cat)


In [ ]:
models = {
    "FP32":     load_model("fp32"),
    "PTQ-INT8": load_model("ptq"),
    "QAT-INT8": load_model("qat"),
}

# map each model to its real saved file on disk (correct size measurement)
model_paths = {"FP32": fp32_path, "PTQ-INT8": ptq_path, "QAT-INT8": qat_path}

summary = []
preds_per_model = {}
for name, m in models.items():
    acc = evaluate(m, test_loader, "cpu")
    size = os.path.getsize(model_paths[name]) / 1e6   # real file size in MB
    lat = cpu_latency_ms(m, runs=100)
    preds_per_model[name] = get_predictions(m, test_loader, "cpu")
    summary.append({"model": name, "test_accuracy_%": round(acc, 2),
                    "size_mb": round(size, 3), "cpu_latency_ms": round(lat, 3)})
    print(f"{name:9s} acc={acc:6.2f}%  size={size:6.2f}MB  latency={lat:6.2f}ms")

# performance CSV
perf_csv = os.path.join(RESULTS, "benchmark_performance.csv")
with open(perf_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["model", "test_accuracy_%", "size_mb", "cpu_latency_ms"])
    w.writeheader(); w.writerows(summary)

# per-sample predictions CSV
true_labels = [test_set[i][1] for i in range(len(test_set))]
pred_csv = os.path.join(RESULTS, "test_predictions.csv")
with open(pred_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["index", "true_label", "FP32_pred", "PTQ_pred", "QAT_pred"])
    for i in range(len(test_set)):
        w.writerow([i, true_labels[i],
                    preds_per_model["FP32"][i],
                    preds_per_model["PTQ-INT8"][i],
                    preds_per_model["QAT-INT8"][i]])

print("\nSaved:", perf_csv)
print("Saved:", pred_csv)

FP32      acc= 90.32%  size= 44.78MB  latency= 19.53ms
PTQ-INT8  acc= 90.27%  size= 11.31MB  latency= 12.56ms
QAT-INT8  acc= 90.69%  size= 11.31MB  latency=  8.75ms

Saved: /content/edge_assignment/q1_quantization/results/benchmark_performance.csv
Saved: /content/edge_assignment/q1_quantization/results/test_predictions.csv


**Question 2**

In [ ]:
import torch.nn.functional as F
import torch.nn.utils.prune as prune

# install the structured-pruning library (dependency-aware channel pruning)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-pruning"], check=True)
import torch_pruning as tp

# Q2 folders
Q2 = os.path.join(ROOT, "q2_pruning")
CKPT2 = os.path.join(Q2, "checkpoints")
RESULTS2 = os.path.join(Q2, "results")
for d in [Q2, CKPT2, RESULTS2]:
    os.makedirs(d, exist_ok=True)

# plain ResNet-18 (same layer names as Q1 model, so Q1 weights load directly)
class PBasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return F.relu(out + identity)

class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64,  2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    def _make_layer(self, planes, blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))
        layers = [PBasicBlock(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(PBasicBlock(self.in_planes, planes))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x); x = torch.flatten(x, 1); x = self.fc(x)
        return x

# load the FP32 weights from Q1 (strict=False ignores Q1's quant stubs, which have no weights)
original = ResNet18(10)
missing, unexpected = original.load_state_dict(torch.load(fp32_path, map_location="cpu"), strict=False)
original_path = os.path.join(CKPT2, "original_fp32.pth")
torch.save(original.state_dict(), original_path)

acc = evaluate(original, test_loader, DEVICE)
print(f"Original FP32 loaded. Test accuracy: {acc:.2f}%")
print("Missing keys:", len(missing), "| Unexpected keys:", len(unexpected), "(expected, both harmless)")

Original FP32 loaded. Test accuracy: 90.32%
Missing keys: 0 | Unexpected keys: 0 (expected, both harmless)


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def count_nonzero(model):
    return sum(int((p != 0).sum()) for p in model.parameters() if p.requires_grad)

def global_sparsity(model):
    total = count_params(model)
    nz = count_nonzero(model)
    return 100.0 * (1 - nz / total)

def finetune(model, epochs, lr, device, tag="FT"):
    model.to(device).train()
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        run = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            run += loss.item()
        sched.step()
        print(f"[{tag}] epoch {ep+1}/{epochs}  loss {run/len(train_loader):.3f}")
    return model

FT_EPOCHS = 3   # fine-tuning epochs to recover accuracy after pruning

In [ ]:
# fresh copy from the original weights
unstructured = ResNet18(10)
unstructured.load_state_dict(torch.load(original_path, map_location="cpu"))
unstructured.to(DEVICE)

# collect all conv + linear weight tensors as pruning targets
prune_targets = [(m, "weight") for m in unstructured.modules()
                 if isinstance(m, (nn.Conv2d, nn.Linear))]

N_ITERATIONS = 3     # iterative pruning rounds
STEP = 0.3           # each round removes 30% of the weights still remaining

for it in range(N_ITERATIONS):
    # global magnitude pruning: removes the smallest-|weight| across ALL layers together
    prune.global_unstructured(prune_targets,
                              pruning_method=prune.L1Unstructured,
                              amount=STEP)
    sp = global_sparsity(unstructured)
    print(f"Iteration {it+1}/{N_ITERATIONS}: sparsity now {sp:.1f}%")
    # fine-tune to recover accuracy lost from this round
    finetune(unstructured, FT_EPOCHS, lr=1e-3, device=DEVICE, tag=f"UNSTRUCT-it{it+1}")

# make pruning permanent: removes the masks and bakes zeros into the weights
for m, n in prune_targets:
    prune.remove(m, n)

unstructured_path = os.path.join(CKPT2, "unstructured_pruned.pth")
torch.save(unstructured.state_dict(), unstructured_path)

u_acc = evaluate(unstructured, test_loader, DEVICE)
print(f"\nUnstructured-pruned accuracy: {u_acc:.2f}%  | sparsity {global_sparsity(unstructured):.1f}%")

Iteration 1/3: sparsity now 0.0%
[UNSTRUCT-it1] epoch 1/3  loss 0.165
[UNSTRUCT-it1] epoch 2/3  loss 0.161
[UNSTRUCT-it1] epoch 3/3  loss 0.157
Iteration 2/3: sparsity now 0.0%
[UNSTRUCT-it2] epoch 1/3  loss 0.158
[UNSTRUCT-it2] epoch 2/3  loss 0.155
[UNSTRUCT-it2] epoch 3/3  loss 0.150
Iteration 3/3: sparsity now 0.0%
[UNSTRUCT-it3] epoch 1/3  loss 0.155
[UNSTRUCT-it3] epoch 2/3  loss 0.151
[UNSTRUCT-it3] epoch 3/3  loss 0.145

Unstructured-pruned accuracy: 90.38%  | sparsity 65.6%


In [ ]:
# fresh copy from the original weights
structured = ResNet18(10)
structured.load_state_dict(torch.load(original_path, map_location="cpu"))
structured.to(DEVICE)

example_inputs = torch.randn(1, 3, 32, 32).to(DEVICE)
params_before = count_params(structured)

# L1 magnitude importance over output channels
imp = tp.importance.MagnitudeImportance(p=1)

pruner = tp.pruner.MagnitudePruner(
    structured,
    example_inputs,
    importance=imp,
    pruning_ratio=0.3,          # remove ~30% of channels
    ignored_layers=[structured.fc],   # never prune the 10-class output layer
)
pruner.step()   # dependency graph removes channels AND fixes all connected layers

params_after = count_params(structured)
print(f"Channels pruned. Params: {params_before:,} -> {params_after:,} "
      f"({100*(1-params_after/params_before):.1f}% fewer)")

# recover accuracy
finetune(structured, FT_EPOCHS, lr=1e-3, device=DEVICE, tag="STRUCT")

# architecture changed, so save the WHOLE model object (not just state_dict)
structured_path = os.path.join(CKPT2, "structured_pruned.pth")
torch.save(structured, structured_path)

s_acc = evaluate(structured, test_loader, DEVICE)
print(f"\nStructured-pruned accuracy: {s_acc:.2f}%  | params {params_after:,}")

Channels pruned. Params: 11,173,962 -> 5,459,866 (51.1% fewer)
[STRUCT] epoch 1/3  loss 0.249
[STRUCT] epoch 2/3  loss 0.217
[STRUCT] epoch 3/3  loss 0.208

Structured-pruned accuracy: 89.84%  | params 5,459,866


In [ ]:
def load_pruned(kind):
    if kind == "original":
        m = ResNet18(10); m.load_state_dict(torch.load(original_path, map_location="cpu")); return m.eval()
    if kind == "unstructured":
        m = ResNet18(10); m.load_state_dict(torch.load(unstructured_path, map_location="cpu")); return m.eval()
    if kind == "structured":
        return torch.load(structured_path, map_location="cpu", weights_only=False).eval()
    raise ValueError(kind)

bench = {
    "Original":     (load_pruned("original"),     original_path),
    "Unstructured": (load_pruned("unstructured"), unstructured_path),
    "Structured":   (load_pruned("structured"),   structured_path),
}

summary2, preds2 = [], {}
for name, (m, path) in bench.items():
    acc = evaluate(m, test_loader, "cpu")
    params = count_params(m)
    sparsity = global_sparsity(m)
    size = os.path.getsize(path) / 1e6
    lat = cpu_latency_ms(m, runs=100)
    preds2[name] = get_predictions(m, test_loader, "cpu")
    summary2.append({"model": name, "test_accuracy_%": round(acc, 2),
                     "params": params, "sparsity_%": round(sparsity, 2),
                     "size_mb": round(size, 3), "cpu_latency_ms": round(lat, 3)})
    print(f"{name:13s} acc={acc:6.2f}%  params={params:>9,}  sparsity={sparsity:5.1f}%  size={size:6.2f}MB  lat={lat:6.2f}ms")

perf2_csv = os.path.join(RESULTS2, "pruning_benchmark.csv")
with open(perf2_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["model","test_accuracy_%","params","sparsity_%","size_mb","cpu_latency_ms"])
    w.writeheader(); w.writerows(summary2)

true_labels = [test_set[i][1] for i in range(len(test_set))]
pred2_csv = os.path.join(RESULTS2, "pruning_predictions.csv")
with open(pred2_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["index","true_label","Original_pred","Unstructured_pred","Structured_pred"])
    for i in range(len(test_set)):
        w.writerow([i, true_labels[i], preds2["Original"][i], preds2["Unstructured"][i], preds2["Structured"][i]])

print("\nSaved:", perf2_csv)
print("Saved:", pred2_csv)

Original      acc= 90.32%  params=11,173,962  sparsity=  0.0%  size= 44.77MB  lat= 20.26ms
Unstructured  acc= 90.38%  params=11,173,962  sparsity= 65.6%  size= 44.78MB  lat= 18.65ms
Structured    acc= 89.84%  params=5,459,866  sparsity=  0.0%  size= 21.92MB  lat= 12.04ms

Saved: /content/edge_assignment/q2_pruning/results/pruning_benchmark.csv
Saved: /content/edge_assignment/q2_pruning/results/pruning_predictions.csv


In [ ]:
!pip install -q onnxscript

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"], check=True)
import onnx, onnxruntime as ort

ONNX_DIR = os.path.join(ROOT, "onnx_models")
os.makedirs(ONNX_DIR, exist_ok=True)

def export_onnx(model, path):
    model.eval().to("cpu")
    dummy = torch.randn(1, 3, 32, 32)
    torch.onnx.export(
        model, dummy, path,
        input_names=["input"], output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=18,                      # avoids the opset-13 conversion errors
    )
    # consolidate external weight files into ONE self-contained .onnx file
    m = onnx.load(path)
    onnx.save_model(m, path, save_as_external_data=False)
    print("exported:", path, f"({os.path.getsize(path)/1e6:.2f} MB)")

# the three float architectures export cleanly
fp32_onnx    = os.path.join(ONNX_DIR, "fp32_baseline.onnx")
unstr_onnx   = os.path.join(ONNX_DIR, "unstructured_pruned.onnx")
struct_onnx  = os.path.join(ONNX_DIR, "structured_pruned.onnx")

export_onnx(load_model("fp32"),           fp32_onnx)      # Q1 FP32
export_onnx(load_pruned("unstructured"),  unstr_onnx)     # Q2 unstructured
export_onnx(load_pruned("structured"),    struct_onnx)    # Q2 structured

# INT8 ONNX via ONNX Runtime static quantization (standard ONNX deployment path)
int8_onnx = os.path.join(ONNX_DIR, "fp32_int8_quantized.onnx")
try:
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
    from onnxruntime.quantization.shape_inference import quant_pre_process

    pre_onnx = os.path.join(ONNX_DIR, "_fp32_prep.onnx")
    quant_pre_process(fp32_onnx, pre_onnx)

    class CalibReader(CalibrationDataReader):
        def __init__(self, loader, n_batches=10):
            self.data = []
            for i, (x, _) in enumerate(loader):
                self.data.append({"input": x.numpy()})
                if i + 1 >= n_batches:
                    break
            self.it = iter(self.data)
        def get_next(self):
            return next(self.it, None)

    quantize_static(pre_onnx, int8_onnx, CalibReader(calib_loader),
                    quant_format=QuantFormat.QDQ, weight_type=QuantType.QInt8)
    print("exported:", int8_onnx, f"({os.path.getsize(int8_onnx)/1e6:.2f} MB)")
except Exception as e:
    print("Static INT8 ONNX skipped (version issue):", e)
    int8_onnx = None

/tmp/ipykernel_388/4128939575.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `QuantResNet18([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `QuantResNet18([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported: /content/edge_assignment/onnx_models/fp32_baseline.onnx (44.76 MB)


/tmp/ipykernel_388/4128939575.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported: /content/edge_assignment/onnx_models/unstructured_pruned.onnx (44.75 MB)


/tmp/ipykernel_388/4128939575.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported: /content/edge_assignment/onnx_models/structured_pruned.onnx (21.90 MB)
exported: /content/edge_assignment/onnx_models/fp32_int8_quantized.onnx (11.27 MB)


In [ ]:
def onnx_eval(path, loader):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    correct = total = 0
    for x, y in loader:
        out = sess.run(None, {"input": x.numpy()})[0]
        correct += int((out.argmax(1) == y.numpy()).sum())
        total += y.size(0)
    return 100.0 * correct / total

def onnx_latency(path, runs=100):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    sess = ort.InferenceSession(path, sess_options=so, providers=["CPUExecutionProvider"])
    x = np.random.randn(1, 3, 32, 32).astype(np.float32)
    for _ in range(10): sess.run(None, {"input": x})
    t0 = time.time()
    for _ in range(runs): sess.run(None, {"input": x})
    return (time.time() - t0) / runs * 1000.0

onnx_models = {"FP32-ONNX": fp32_onnx, "Unstructured-ONNX": unstr_onnx, "Structured-ONNX": struct_onnx}
if int8_onnx: onnx_models["INT8-ONNX"] = int8_onnx

onnx_rows = []
for name, path in onnx_models.items():
    acc = onnx_eval(path, test_loader)
    size = os.path.getsize(path) / 1e6
    lat = onnx_latency(path)
    onnx_rows.append({"model": name, "test_accuracy_%": round(acc, 2),
                      "size_mb": round(size, 3), "cpu_latency_ms": round(lat, 3)})
    print(f"{name:18s} acc={acc:6.2f}%  size={size:6.2f}MB  lat={lat:6.2f}ms")

onnx_csv = os.path.join(ONNX_DIR, "onnx_benchmark.csv")
with open(onnx_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["model", "test_accuracy_%", "size_mb", "cpu_latency_ms"])
    w.writeheader(); w.writerows(onnx_rows)
print("\nSaved:", onnx_csv)

FP32-ONNX          acc= 90.32%  size= 44.76MB  lat= 12.56ms
Unstructured-ONNX  acc= 90.38%  size= 44.75MB  lat= 12.63ms
Structured-ONNX    acc= 89.84%  size= 21.90MB  lat= 10.87ms
INT8-ONNX          acc= 90.46%  size= 11.27MB  lat= 13.66ms

Saved: /content/edge_assignment/onnx_models/onnx_benchmark.csv
